This notebook uses a LLM to answer questions 

In [1]:
import platform
import requests

# For the paper analyser
import torch
import transformers
import argparse
import logging
import json
import os
import accelerate

c:\Users\thiba\anaconda3\envs\evidence_db2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#  Get ChromaDB Collection

In [2]:
!pip install chromadb
#!pip install sentence_transformers

In [3]:
#get chromaDB and collection (collection must have been created and populated previously)
import chromadb
from chromadb.utils import embedding_functions

CHROMA_DATA_PATH = "chroma_data2/"
COLLECTION_NAME = "searchable_db_collection"

client = chromadb.PersistentClient(path=CHROMA_DATA_PATH)
collection = client.get_collection(name="searchable_db_collection")

### USE LLama to answer a query

In [5]:
from openai import OpenAI

client = OpenAI(
        api_key = "W2oF2Q2NLaTmqj7LGOiJwp9Pdi47Rhhn",
        base_url="https://api.deepinfra.com/v1/openai",
    )

SYSTEM_MSG  = "You are a helpful systematic reviewing assistant"

def generateFromPrompt(promptStr,maxTokens=100):
    messages=[
    {"role": "system", "content": SYSTEM_MSG},
    {"role": "user", "content": promptStr}
    ]
    completion = client.chat.completions.create(
    model="meta-llama/Meta-Llama-3.1-70B-Instruct",
    messages=messages)
    response=completion.choices[0].message.content
    return(response)

In [6]:
prompt = "Please answer the following question using the following paper titles and abstracts."
query = "What are the expected outcomes for a middle-aged man with prostate cancer stage III? What are possible treatments?"
NB_PAPERS_LLM = 3


query_results = collection.query(
    query_texts=[query],
    n_results=NB_PAPERS_LLM,
)

title_and_abst = ",".join(query_results["documents"][0])

answer = generateFromPrompt(prompt + query + title_and_abst)

print("Retrieved from ",query_results["ids"][0], \
      "\n Titles: \n", {query_results['metadatas'][0][i]['titles'] for i in range(NB_PAPERS_LLM)}, \
      "\n \n Answer:",answer,\
      "\n\nTitle and abstract:",query_results['documents'][0])

Retrieved from  ['3873', '9212', '10432'] 
 Titles: 
 {'The Future of Advanced Prostate Cancer Treatment', 'Retrospective Analysis of Clinico-Epidimological Factors in Prostatic Cancer', 'Estimates of survival from incurable prostate cancer need to be revised upwards'} 
 
 Answer: Based on the provided paper titles and abstracts, I'll provide an answer to your question.

**Expected outcomes for a middle-aged man with prostate cancer stage III:**

Unfortunately, the provided abstracts do not specifically address the expected outcomes for a middle-aged man with prostate cancer stage III. However, we can infer some information from the studies.

The second abstract, "Retrospective Analysis of Clinico-Epidimological Factors in Prostatic Cancer," reports on a study of 101 patients with localized or metastatic prostate cancer. While the study doesn't specifically focus on stage III patients, it provides some general insights into the survival rates of prostate cancer patients. The study foun

In [7]:
query_results

{'ids': [['3873', '9212', '10432']],
 'embeddings': None,
 'documents': [['Estimates of survival from incurable prostate cancer need to be revised upwards Men with advanced prostate cancer are living for an average of two years longer than they did a decade ago, an analysis has found.\n\nThe researchers said that the findings meant that the models used to predict survival among men with metastatic, castration resistant prostate cancer needed to be updated upwards to reflect the impact of new treatments.\n\nThe study looked at 442 men …',
   'The Future of Advanced Prostate Cancer Treatment <p />',
   'Retrospective Analysis of Clinico-Epidimological Factors in Prostatic Cancer Background: Prostate cancer is the second most common cancer in men and the seventh leading cause of male cancer death worldwide.It is a highly heterogenous disease with great variability in its clinical course.Treatment options vary depending on age, stage, and grade of cancer, as well as other medical condition